In [6]:
import xarray as xr
import numpy as np

era5_dir="/hkfs/work/workspace/scratch/xo8179-neural_lam/data/data/global_era5_1980_2022_6h-128x64_equiangular_with_poles_conservative/fields.zarr"
ds = xr.open_dataset(era5_dir)
print(ds)

<xarray.Dataset> Size: 167GB
Dimensions:                  (time: 61400, longitude: 128, latitude: 64,
                              level: 13)
Coordinates:
  * latitude                 (latitude) float64 512B -90.0 -87.14 ... 87.14 90.0
  * level                    (level) int64 104B 50 100 150 200 ... 850 925 1000
  * longitude                (longitude) float64 1kB 0.0 2.812 ... 354.4 357.2
  * time                     (time) datetime64[ns] 491kB 1979-12-23 ... 2021-...
Data variables: (12/13)
    10m_u_component_of_wind  (time, longitude, latitude) float32 2GB ...
    10m_v_component_of_wind  (time, longitude, latitude) float32 2GB ...
    2m_temperature           (time, longitude, latitude) float32 2GB ...
    geopotential             (time, level, longitude, latitude) float32 26GB ...
    geopotential_at_surface  (longitude, latitude) float32 33kB ...
    land_sea_mask            (longitude, latitude) float32 33kB ...
    ...                       ...
    specific_humidity        

In [2]:
import var_dicts as d
from pathlib import Path

# Atmospheric variables to process
atm_vars = ["t", "u", "v", "w", "z", "q"]

# Surface variables
surface_vars = ["2t", "10u", "10v", "msl", "tp"]

# Your working directory
data_dir = Path("/hkfs/work/workspace/scratch/xo8179-nextgems_regridded/try3/")

out_name = "NextGEMS_1990_2020_6hourly_128x64_equiangular_with_poles_conservative.zarr"

t_chunk = 50

In [4]:
ds_something = xr.open_zarr(str(data_dir / f"3D_nextgems_1990s_6hourly_128x64_t.zarr"))
print(ds_something)

<xarray.Dataset> Size: 12GB
Dimensions:    (latitude: 64, level: 13, longitude: 128, time: 14608)
Coordinates:
  * latitude   (latitude) float64 512B -90.0 -87.14 -84.29 ... 84.29 87.14 90.0
  * level      (level) int64 104B 50 100 150 200 250 ... 600 700 850 925 1000
  * longitude  (longitude) float64 1kB 0.0 2.812 5.625 ... 351.6 354.4 357.2
  * time       (time) datetime64[ns] 117kB 1990-01-01 ... 1999-12-31T18:00:00
Data variables:
    t          (time, level, longitude, latitude) float64 12GB dask.array<chunksize=(1, 13, 128, 64), meta=np.ndarray>


In [8]:

# Open and concatenate atmospheric variables
atm_datasets = []

for var in atm_vars:
    decade_files = [
        data_dir / f"3D_nextgems_1990s_6hourly_128x64_{var}.zarr",
        data_dir / f"3D_nextgems_2000s_6hourly_128x64_{var}.zarr",
        data_dir / f"3D_nextgems_2010s_6hourly_128x64_{var}.zarr",
    ]
    ds_decades = []

    for f in decade_files:
        # Open as xarray datasets
        ds_var_decade = xr.open_zarr(str(f))

        ds_var_decade = ds_var_decade.astype(np.float32)
            
        var_rename_dict_filtered = {
            k: v for k, v in d.var_rename_dict.items()
            if k in ds_var_decade.variables or k in ds_var_decade.dims
        }
        ds_var_decade = ds_var_decade.rename(var_rename_dict_filtered)
        ds_decades.append(ds_var_decade)
    
    # Concatenate along 'time'
    ds_concat = xr.concat(ds_decades, dim="time")
    
    atm_datasets.append(ds_concat)
    # print(ds_concat)

# Merge all atmospheric variables into one dataset
ds_atm = xr.merge(atm_datasets)

# ds_atm = ds_atm.rename(var_rename_dict)

ds_atm = ds_atm.chunk({"time": t_chunk, "level": 13, "longitude": 128, "latitude": 64})
encoding = {var: {"chunks": (t_chunk, 13, 128, 64)} for var in ds_atm.data_vars}
ds_atm.to_zarr(data_dir.joinpath(out_name), mode="w", encoding=encoding)


In [11]:

# Open surface variables
surface_files = [
    data_dir / "surface_nextgems_1990s_2020_6hourly_0.25deg_surface_2t.zarr",
    data_dir / "surface_nextgems_1990s_2020_6hourly_0.25deg_surface_10u.zarr",
    data_dir / "surface_nextgems_1990s_2020_6hourly_0.25deg_surface_10v.zarr",
    data_dir / "surface_nextgems_1990s_2020_6hourly_0.25deg_surface_msl.zarr",
    data_dir / "surface_nextgems_1990s_2020_6hourly_0.25deg_total_precipitation.zarr"
]

surface_datasets = []

for f in surface_files:
    # Open as xarray datasets
    ds_var_surface = xr.open_zarr(str(f)).isel(time=slice(0,-1))

    ds_var_surface = ds_var_surface.astype(np.float32)
        
    var_rename_dict_filtered = {
        k: v for k, v in d.var_rename_dict.items()
        if k in ds_var_surface.variables or k in ds_var_surface.dims
    }
    ds_var_surface = ds_var_surface.rename(var_rename_dict_filtered)
    surface_datasets.append(ds_var_surface)

for dataset in surface_datasets:
    print(dataset)
# Merge all surface variables into one dataset
ds_surface = xr.merge(surface_datasets)

# ds_surface = ds_surface.rename(var_rename_dict)

# Apply atmospheric chunking
ds_surface = ds_surface.chunk({"time": t_chunk, "longitude": 128, "latitude": 64})


encoding = {var: {"chunks": (t_chunk, 128, 64)} for var in ds_surface.data_vars}
ds_surface.to_zarr(data_dir.joinpath(out_name), mode="a", encoding=encoding)

<xarray.Dataset> Size: 1GB
Dimensions:         (time: 43828, longitude: 128, latitude: 64)
Coordinates:
  * latitude        (latitude) float64 512B -90.0 -87.14 -84.29 ... 87.14 90.0
  * longitude       (longitude) float64 1kB 0.0 2.812 5.625 ... 354.4 357.2
  * time            (time) datetime64[ns] 351kB 1990-01-01 ... 2019-12-31T18:...
Data variables:
    2m_temperature  (time, longitude, latitude) float32 1GB dask.array<chunksize=(1, 128, 64), meta=np.ndarray>
<xarray.Dataset> Size: 1GB
Dimensions:                  (time: 43828, longitude: 128, latitude: 64)
Coordinates:
  * latitude                 (latitude) float64 512B -90.0 -87.14 ... 87.14 90.0
  * longitude                (longitude) float64 1kB 0.0 2.812 ... 354.4 357.2
  * time                     (time) datetime64[ns] 351kB 1990-01-01 ... 2019-...
Data variables:
    10m_u_component_of_wind  (time, longitude, latitude) float32 1GB dask.array<chunksize=(1, 128, 64), meta=np.ndarray>
<xarray.Dataset> Size: 1GB
Dimensions:   

In [12]:

ds_era5 = xr.open_dataset(era5_dir)
remaining = ds_era5[["land_sea_mask", "geopotential_at_surface"]]

remaining.to_zarr(data_dir.joinpath(out_name), mode="a")


In [6]:
ds = xr.open_dataset(data_dir.joinpath("NextGEMS_1990_2020_6hourly_128x64_merged.zarr"))
print(ds)
print(data_dir.joinpath("NextGEMS_1990_2020_6hourly_128x64_merged.zarr"))

<xarray.Dataset> Size: 119GB
Dimensions:                  (time: 43828, longitude: 128, latitude: 64,
                              level: 13)
Coordinates:
  * latitude                 (latitude) float64 512B -90.0 -87.14 ... 87.14 90.0
  * level                    (level) int64 104B 50 100 150 200 ... 850 925 1000
  * longitude                (longitude) float64 1kB 0.0 2.812 ... 354.4 357.2
  * time                     (time) datetime64[ns] 351kB 1990-01-01 ... 2019-...
Data variables: (12/13)
    10m_u_component_of_wind  (time, longitude, latitude) float32 1GB ...
    10m_v_component_of_wind  (time, longitude, latitude) float32 1GB ...
    2m_temperature           (time, longitude, latitude) float32 1GB ...
    geopotential             (time, level, longitude, latitude) float32 19GB ...
    geopotential_at_surface  (longitude, latitude) float32 33kB ...
    land_sea_mask            (longitude, latitude) float32 33kB ...
    ...                       ...
    specific_humidity        